In [31]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [32]:
data = pd.read_csv("clean_data.csv")

In [33]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16942 entries, 0 to 16941
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Net_Metrekare       16942 non-null  int64  
 1   Brüt_Metrekare      16942 non-null  float64
 2   Oda_Sayısı          16942 non-null  float64
 3   Bulunduğu_Kat       16942 non-null  object 
 4   Binanın_Yaşı        16942 non-null  object 
 5   Isıtma_Tipi         16942 non-null  object 
 6   Fiyat               16942 non-null  float64
 7   Şehir               16942 non-null  object 
 8   Binanın_Kat_Sayısı  16942 non-null  int64  
 9   Kullanım_Durumu     16942 non-null  object 
 10  Banyo_Sayısı        16942 non-null  float64
dtypes: float64(4), int64(2), object(5)
memory usage: 1.4+ MB


## Model oluşturmaya hazırlık

Veri setimiziden hedef sütunu ayırma

In [34]:
x = data.drop("Fiyat",axis=1)
y = data["Fiyat"]


Kategorik verileri encode etme

In [35]:
kategorik_veri = x.select_dtypes(include=["object"]).columns
x = pd.get_dummies(x,columns=kategorik_veri,drop_first=True)

x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16942 entries, 0 to 16941
Columns: 114 entries, Net_Metrekare to Kullanım_Durumu_Mülk Sahibi Oturuyor
dtypes: bool(109), float64(3), int64(2)
memory usage: 2.4 MB


Eğitim ve Test verisi olarak ayırma

In [36]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.33,random_state=42)

Ölçeklendirme 

Scaling önemli olan modeller:

    Linear Regression

    SVR

    KNN

    Logistic Regression

Scaling gerekmeyen modeller:

    Decision Tree

    Random Forest

    XGBoost

In [37]:
sc = StandardScaler()

x_train_scaled = sc.fit_transform(x_train)
x_test_scaled = sc.transform(x_test)

## Model oluşturma

Kütüphaneler

In [38]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


## Model Seçme

### Metrik anlamları:
##### 1. MAE (Ortalama Mutlak Hata)

###### Detay: Tahminlerin ile gerçek değerler arasındaki farkların basit ortalamasıdır. Hataların negatif veya pozitif olması fark etmez.

###### Örnek: Bir araba fiyatı tahmin modelinde MAE değerin 10.000 ise, modelin arabaların fiyatını ortalama 10.000 TL eksik veya fazla tahmin ediyor demektir.

##### 2. RMSE (Kök Ortalama Kare Hata)

###### Detay: Hataların önce karesi alınıp sonra ortalamasının karekökü bulunduğu için, modelin yaptığı büyük hatalara karşı çok daha hassastır ve bunları yüksek skorla cezalandırır.

###### Örnek: 100 arabanın fiyatını 1.000 TL hatayla, 1 arabanın fiyatını ise 500.000 TL hatayla tahmin edersen; MAE bu durumu pek umursamaz ama RMSE o tek büyük hata yüzünden anında fırlar. Modelin tutarsızlıklarını görmek için harikadır.

##### 3. R² (R-Kare / Modelin Açıklama Gücü)

###### Detay: Genellikle 0 ile 1 arasında değer alır. Elindeki girdilerin (özelliklerin), tahmin etmeye çalıştığın hedefteki değişimi yüzde kaç oranında açıklayabildiğini gösterir. 1'e ne kadar yakınsa model o kadar başarılıdır.

###### Örnek: R² skorun 0.85 ise, araba fiyatlarındaki değişikliğin %85'ini modelindeki verilerle (kilometre, yıl, hasar durumu vb.) doğru bir şekilde yakalayabiliyorsun demektir.

In [39]:
models = {
    "RandomForest": RandomForestRegressor(),
    "SVR" : SVR(),
    "GradientBoostingRegressor" : GradientBoostingRegressor(),
    "XGBRegressor" : XGBRegressor()

}

results = []

for name, model in models.items():

    # SVR scaled veri kullanır
    if name in ["SVR"]:
        model.fit(x_train_scaled, y_train)
        y_pred = model.predict(x_test_scaled)
    else:
        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)

    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    results.append([
        name,
        round(r2,2),
        round(mae/1000,0),
        round(rmse/1000000,2)
    ])

    print(f"{name} R2: {r2:.2f}")
    print(f"{name} MAE: {mae/1000:.0f}k")
    print(f"{name} RMSE: {rmse/1000000:.2f}M")
    print("----------------------")


# Decision Tree (ayrı tutmana gerek yok ama kalsın)
dt = DecisionTreeRegressor(max_depth=3)
dt.fit(x_train, y_train)
dt_pred = dt.predict(x_test)

r2 = r2_score(y_test, dt_pred)
mae = mean_absolute_error(y_test, dt_pred)
rmse = np.sqrt(mean_squared_error(y_test, dt_pred))

results.append([
    "DecisionTree",
    round(r2,2),
    round(mae/1000,0),
    round(rmse/1000000,2)
])

# DataFrame
results_df = pd.DataFrame(results, columns=["Model","R2","MAE","RMSE"])

# Sırala
results_df = results_df.sort_values(by="R2", ascending=False)

results_df
   

RandomForest R2: 0.57
RandomForest MAE: 534k
RandomForest RMSE: 0.85M
----------------------
SVR R2: -0.05
SVR MAE: 910k
SVR RMSE: 1.33M
----------------------
GradientBoostingRegressor R2: 0.56
GradientBoostingRegressor MAE: 566k
GradientBoostingRegressor RMSE: 0.86M
----------------------
XGBRegressor R2: 0.60
XGBRegressor MAE: 519k
XGBRegressor RMSE: 0.82M
----------------------


,Model,R2,MAE,RMSE
3,XGBRegressor,0.60,519.0,0.82
0,RandomForest,0.57,534.0,0.85
2,GradientBoostingRegressor,0.56,566.0,0.86
4,DecisionTree,0.37,723.0,1.03
1,SVR,-0.05,910.0,1.33


# EN İYİ MODEL XGBRegressor


In [40]:
best_model = XGBRegressor()
best_model.fit(x_train, y_train)

y_pred_best = best_model.predict(x_test)